In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from skimage import color
from scipy import ndimage

### Check that we have masks available

In [2]:
masks_path = "masks/"
imgs_path = "imgs/"

In [8]:
def read_masks(folder):
    masks = []

    for filename in os.listdir(folder):
        if filename.endswith(".png"):
            mask = imageio.imread(os.path.join(folder, filename))
            masks.append(mask)

    return masks

In [9]:
masks = read_masks(masks_path)

In [10]:
len(masks)

"117"

In [38]:
def asymmetry2(mask):
    # Left–right asymmetry
    flipped_lr = np.fliplr(mask)
    overlap_lr = np.sum(mask & flipped_lr) / np.sum(mask | flipped_lr)
    asym_lr = 1 - overlap_lr

    # Top–bottom asymmetry
    flipped_tb = np.flipud(mask)
    overlap_tb = np.sum(mask & flipped_tb) / np.sum(mask | flipped_tb)
    asym_tb = 1 - overlap_tb

    # Combine using Noisy-OR (high if either is high)
    final = 1 - (1 - asym_lr) * (1 - asym_tb)

    return final


#### Example mole

In [35]:
plt.figure(figsize = (3,3))
plt.imshow(masks[3]);

### Asymmetry: Noisy-OR

This method calculates the overlap between the left/right halves and separately the up/down halves of the mask, giving two asymmetry scores $v$ (vertical) and $h$ (horizontal). These are combined using the Noisy-OR formula:

$$\\text{asymmetry} = 1 - (1 - v)(1 - h)$$

The advantage of this over a simple average is that if either direction is highly asymmetric, the overall score is high. With an average, a very irregular shape could still score low if it happens to be symmetric in the other direction.

In [31]:
asymmetry2(masks[3])

"np.float64(0.6748506422491927)"

## Colour Variation (C in ABCD)

Colour variation is one of the key features used to identify potentially malignant melanomas. A benign mole typically has a uniform colour, while a suspicious one might have multiple different colours (shades of brown, red, black, white, or blue). The idea here is to measure how much colour variation exists within the lesion area defined by the mask.

### Reading images

We also need to load the actual RGB images (not just the masks) since we need the colour information.

In [40]:
def read_imgs(folder):
    imgs = []
    for filename in sorted(os.listdir(folder)):
        if filename.endswith(".png") or filename.endswith(".jpg"):
            img = imageio.imread(os.path.join(folder, filename))
            imgs.append(img)
    return imgs

imgs = read_imgs(imgs_path)

### Colour variation using standard deviation in LAB space

The idea is to:
1. Convert the image from RGB to LAB colour space (LAB is better for measuring perceptual colour differences)
2. Extract only the pixels that are inside the lesion (where mask == 1)
3. Calculate the standard deviation of each channel (L, A, B) for those pixels
4. Combine the standard deviations into one score

A higher score means more colour variation = more suspicious

### The LAB Colour Space

LAB (also written as L\*a\*b\*) is a colour space designed to be **perceptually uniform** this means that equal numerical distances correspond to roughly equal perceived colour differences. This makes it much better suited for measuring colour variation than RGB.

The three channels are:

- **L (Lightness):** Ranges from 0 (black) to 100 (white). Captures how light or dark a colour is, independently of its hue.
- **A (Green–Red axis):** Negative values represent green tones, positive values represent red/magenta tones. In skin lesions, variation along this axis can reflect the presence of redness or erythema versus darker pigmentation.
- **B (Blue–Yellow axis):** Negative values represent blue tones, positive values represent yellow tones. In melanoma analysis, this channel is relevant because suspicious lesions may contain blue-grey areas (a feature known as blue-white veil) alongside yellow-brown pigmentation.

By computing the standard deviation of each channel separately and averaging them, we capture overall perceptual colour spread within the lesion. A uniformly coloured mole will have low standard deviations across all three channels, while a multi-coloured suspicious lesion will show high variation in one or more channels.

In [41]:
def colour_variation(img, mask):
    
    img_float = img.astype(np.float64) / 255.0
     
    img_lab = color.rgb2lab(img_float)
    
    mask_bool = mask.astype(bool)
    
    l_vals = img_lab[:, :, 0][mask_bool]
    a_vals = img_lab[:, :, 1][mask_bool]
    b_vals = img_lab[:, :, 2][mask_bool]
    
 
    std_l = np.std(l_vals)
    std_a = np.std(a_vals)
    std_b = np.std(b_vals)
    
    
    score = (std_l + std_a + std_b) / 3
    
    return score

### Testing the colour variation function

The function is tested on mole index 3  used the same example used in the asymmetry section.

In [42]:
plt.figure(figsize=(3, 3))
plt.imshow(imgs[3])
plt.title("Original image")
plt.axis('off');

In [43]:
colour_variation(imgs[3], masks[3])

### Compute colour variation for all images

In [44]:
colour_scores = [colour_variation(imgs[i], masks[i]) for i in range(len(masks))]

plt.figure(figsize=(8, 4))
plt.hist(colour_scores, bins=20)
plt.xlabel('Colour variation score')
plt.ylabel('Count')
plt.title('Distribution of colour variation scores across all lesions')
plt.tight_layout()
plt.show()

## Border Irregularity (B in ABCD)

Border irregularity measures how uneven or jagged the border of the lesion is. A smooth, regular border usually suggests a benign lesion, while an irregular, notched border is more associated with malignancy.

Two approaches are explored below.

### Approach 1: Compactness

A classic way to measure shape irregularity is the **compactness** (or circularity) score:

$$\text{compactness} = \frac{P^2}{4\pi A}$$

where $P$ is the perimeter and $A$ is the area of the lesion.

- A perfect circle has compactness = 1
- Any other shape has compactness > 1
- The more irregular the border, the higher the compactness score

To get border irregularity (so higher = more irregular, like our other scores), we can just use compactness directly since it's already >= 1 for irregular shapes. Or we can normalise it somehow.

In [50]:
def border_irregularity_compactness(mask):
    
    binary_mask = (mask > 0).astype(np.uint8)
      
    area = np.sum(binary_mask)
     
    eroded = ndimage.binary_erosion(binary_mask)
    border = binary_mask - eroded.astype(np.uint8)
    perimeter = np.sum(border)
    
    compactness = (perimeter ** 2) / (4 * np.pi * area)
    
    return compactness

In [51]:

border_irregularity_compactness(masks[3])

### Approach 2: Border irregularity using radial distance variance

Another approach is to:
1. Find the centroid of the lesion
2. Measure the distance from the centroid to each border pixel
3. Calculate the variance of those distances

If the border is smooth and circular the distances will all be similar (low variance). If the border is jagged and irregular the distances will vary a lot (high variance).

We normalise by the mean distance so that the score doesn't just reflect the size of the lesion.

In [52]:
def border_irregularity_radial(mask):
    binary_mask = (mask > 0).astype(np.uint8)
    
    eroded = ndimage.binary_erosion(binary_mask)
    border = binary_mask - eroded.astype(np.uint8)
    
    border_coords = np.argwhere(border > 0)  # returns (row, col) pairs
    
    if len(border_coords) == 0:
        return 0.0
     
    mask_coords = np.argwhere(binary_mask > 0)
    centroid = mask_coords.mean(axis=0)  # (row, col)
    
    
    distances = np.sqrt(np.sum((border_coords - centroid) ** 2, axis=1))
       
    mean_dist = np.mean(distances)
    score = np.std(distances) / mean_dist
    
    return score

In [53]:
border_irregularity_radial(masks[3])

### Comparing the two border approaches on all masks

In [54]:
compact_scores = [border_irregularity_compactness(m) for m in masks]
radial_scores = [border_irregularity_radial(m) for m in masks]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(compact_scores, bins=20)
axes[0].set_title('Compactness scores')
axes[0].set_xlabel('Score (higher = more irregular)')
axes[0].set_ylabel('Count')

axes[1].hist(radial_scores, bins=20)
axes[1].set_title('Radial distance variance scores')
axes[1].set_xlabel('Score (higher = more irregular)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

### Visualise border extraction on an example

Just to double check the border extraction is working correctly

In [55]:
binary_mask = (masks[3] > 0).astype(np.uint8)
eroded = ndimage.binary_erosion(binary_mask)
border = binary_mask - eroded.astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))

axes[0].imshow(binary_mask, cmap='gray')
axes[0].set_title('Mask')
axes[0].axis('off')

axes[1].imshow(border, cmap='gray')
axes[1].set_title('Border only')
axes[1].axis('off')

overlay = imgs[3].copy()
overlay[border > 0] = [255, 0, 0]  # mark border in red
axes[2].imshow(overlay)
axes[2].set_title('Border overlaid on image')
axes[2].axis('off')

plt.tight_layout()
plt.show()